# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python) 

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [1]:
# importar librerías
import pandas as pd

In [2]:
# cargar archivos
orders = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv')
catalog = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv')
marketing = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv')

In [3]:
# explorar dataset orders
orders.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25100 entries, 0 to 25099
Data columns (total 12 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_pedido           25100 non-null  object 
 1   id_usuario          25100 non-null  object 
 2   fecha_hora_pedido   25100 non-null  object 
 3   pais                24800 non-null  object 
 4   dispositivo         25080 non-null  object 
 5   fuente_referencia   25070 non-null  object 
 6   nombre_producto     25070 non-null  object 
 7   categoria_producto  25020 non-null  object 
 8   cantidad            25050 non-null  float64
 9   precio_unitario     25050 non-null  float64
 10  monto_descuento     25050 non-null  float64
 11  monto_total         25100 non-null  float64
dtypes: float64(4), object(8)
memory usage: 2.3+ MB


In [4]:
#visualizamos las primeras filas de dataset orders
orders.head()

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28


In [5]:
#exploramos medidas estadísticas en columnas numéricas de dataset orders
orders.describe()

,cantidad,precio_unitario,monto_descuento,monto_total
count,25050.000000,25050.000000,25050.000000,2.510000e+04
mean,7.092735,259.305549,4.500798,2.072680e+03
std,296.277003,138.726461,5.223010,9.894995e+04
min,-2.000000,20.030000,0.000000,-4.926500e+02
25%,1.000000,138.377500,0.000000,1.805075e+02
50%,2.000000,258.715000,0.000000,3.417500e+02
75%,2.000000,380.332500,10.000000,5.185800e+02
max,20000.000000,499.960000,15.000000,8.840200e+06


In [6]:
#explorar variables categóricas y como se distribuyen de dataset orders

vars_categoricas_orders =['pais','dispositivo','fuente_referencia','nombre_producto','categoria_producto',]
for var in vars_categoricas_orders:
    print(f"\n=== Distribución de {var} ===")
    print(orders[var].value_counts())



=== Distribución de pais ===
Colombia     7520
Mexico       7502
Argentina    7291
mexico        865
colombia      823
argentina     799
Name: pais, dtype: int64

=== Distribución de dispositivo ===
desktop    12759
mobile     12321
Name: dispositivo, dtype: int64

=== Distribución de fuente_referencia ===
social         8428
organic        8329
paid_search    8313
Name: fuente_referencia, dtype: int64

=== Distribución de nombre_producto ===
Vacuum-Pro-Black        4199
Blender-XL-Red          4195
Jacket-Winter-M         4192
Sneakers-Urban-42       4160
Laptop-Gaming-16GB      2794
Tablet-Standard-64GB    2780
Phone-Pro-128GB         2750
Name: nombre_producto, dtype: int64

=== Distribución de categoria_producto ===
Hogar          8385
Moda           8323
Electronica    8312
Name: categoria_producto, dtype: int64


---

In [7]:
#explorar dataset catalog
catalog.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7 entries, 0 to 6
Data columns (total 4 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   nombre_producto     7 non-null      object 
 1   categoria_producto  7 non-null      object 
 2   costo_unitario      7 non-null      float64
 3   proveedor           7 non-null      object 
dtypes: float64(1), object(3)
memory usage: 352.0+ bytes


In [8]:
# mostrar las siete filas dataset catalog
catalog.head(7)

,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"
5,Sneakers-Urban-42,Moda,17.21,Greene-Smith
6,Jacket-Winter-M,Moda,189.31,Mcmillan-Rhodes


In [9]:
#exploramos medidas estadísticas en columnas numéricas de dataset catalog
catalog.describe()

,costo_unitario
count,7.000000
mean,102.252857
std,111.011563
min,10.120000
25%,16.905000
50%,25.210000
75%,182.975000
max,280.680000


In [10]:

#exploramos variables categóricas y como se distribuyen en dataset catalog
vars_categoricas_catalog =['nombre_producto','categoria_producto','proveedor']
for var in vars_categoricas_catalog:
    print(f"\n=== Distribución de {var} ===")
    print(catalog[var].value_counts())



=== Distribución de nombre_producto ===
Vacuum-Pro-Black        1
Phone-Pro-128GB         1
Tablet-Standard-64GB    1
Laptop-Gaming-16GB      1
Sneakers-Urban-42       1
Blender-XL-Red          1
Jacket-Winter-M         1
Name: nombre_producto, dtype: int64

=== Distribución de categoria_producto ===
Electrónica    3
Moda           2
Hogar          2
Name: categoria_producto, dtype: int64

=== Distribución de proveedor ===
Bowers LLC                 1
Fuller, Pena and Myers     1
Long-Reid                  1
King Ltd                   1
Rivera, Carr and Finley    1
Mcmillan-Rhodes            1
Greene-Smith               1
Name: proveedor, dtype: int64


In [11]:
# explorar dataset marketing
marketing.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB


In [12]:
# mostrar primeras cinco filas dataset marketing
marketing.head()

,fecha,pais,id_campaña,canal,gasto
0,2025-01-01,Mexico,organic_Mexico,organic,2446.25
1,2025-01-01,Mexico,paid_search_Mexico,paid_search,2704.34
2,2025-01-01,Mexico,social_Mexico,social,2045.01
3,2025-01-01,Colombia,organic_Colombia,organic,2597.21
4,2025-01-01,Colombia,paid_search_Colombia,paid_search,1771.40


In [13]:
#exploramos medidas estadísticas en columnas numéricas de dataset marketing
marketing.describe()

,gasto
count,1620.00000
mean,1772.74292
std,734.43294
min,501.11000
25%,1128.03000
50%,1782.42500
75%,2420.68500
max,2999.36000


In [14]:
#exploramos variables categóricas y como se distribuyen en dataset marketing
vars_categoricas_marketing =['pais','canal']
for var in vars_categoricas_marketing:
    print(f"\n=== Distribución de {var} ===")
    print(marketing[var].value_counts())


=== Distribución de pais ===
Mexico       540
Argentina    540
Colombia     540
Name: pais, dtype: int64

=== Distribución de canal ===
paid_search    507
organic        506
social         506
Name: canal, dtype: int64


### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.


Se revisan los 3 datasets
- Creamos una copia de los tres dataset para trabajar.
- Validar y convertir fechas al formato correcto.  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos.  
- Eliminar duplicados.  
- Revisar variables categóricas.
- Convertimos a número columnas numéricas. 

In [15]:
# creamos copia de los datasets
orders_work = orders.copy()
catalog_work = catalog.copy()
marketing_work = marketing.copy()

In [16]:
# validar y convertir fechas a formato datetime dataset orders
# errors='coerce' para convertir fechas inválidas en NaT
orders_work['fecha_hora_pedido']=pd.to_datetime(orders_work['fecha_hora_pedido'],errors='coerce')

In [17]:
# validar y convertir fechas a formato datetime dataset marketing
# errors='coerce' para convertir fechas inválidas en NaT
marketing_work['fecha']=pd.to_datetime(marketing_work['fecha'],errors='coerce')

In [18]:
# de acuerdo a información obtenida con orders.info(), verificamos cantidad de valores inválidos (<=0) en columna cantidad de dataset orders
cantidad_invalidos = (orders_work['cantidad']<=0).sum()
print(f"Cantidad de valores inválidos: {cantidad_invalidos}")

Cantidad de valores inválidos: 4


In [19]:
# marcamos los inválidos (<=0) con NaN 
orders_work.loc[orders['cantidad'] <= 0, 'cantidad'] = pd.NA

In [20]:
# comprobar si cantidad calculada sigue la fórmula cantidad= valor_sin_dcto/precio_unitario
orders_work['valor_sin_dcto']= (orders_work['cantidad']*orders_work['precio_unitario'])
orders_work['cantidad_calculada']=(orders_work['valor_sin_dcto']/orders_work['precio_unitario'])

#verificamos las primeras filas
orders_work[['valor_sin_dcto','precio_unitario','cantidad','cantidad_calculada']].head()

,valor_sin_dcto,precio_unitario,cantidad,cantidad_calculada
0,665.38,332.69,2.0,2.0
1,176.86,176.86,1.0,1.0
2,205.98,102.99,2.0,2.0
3,257.87,257.87,1.0,1.0
4,336.28,336.28,1.0,1.0


In [21]:
# imputamos cantidad siguiendo el cálculo
orders_work['cantidad']=orders_work['cantidad'].fillna(orders_work['valor_sin_dcto']/orders_work['precio_unitario'])

In [22]:
# revisar la imputación
cantidad_menores_igual_cero = len(orders_work[orders_work['cantidad'] <= 0])
print(f"Cantidad de valores <= 0: {cantidad_menores_igual_cero}")


Cantidad de valores <= 0: 0


In [23]:
# de acuerdo a información obtenida con orders.info(), verificamos cantidad de valores inválidos (<=0) en columna monto_total de dataset orders
cantidad_invalidos_monto = (orders_work['monto_total']<=0).sum()
print(f"Cantidad de valores inválidos: {cantidad_invalidos_monto}")

Cantidad de valores inválidos: 4


In [24]:
# marcamos los inválidos (<=0) con NaN 
orders_work.loc[orders_work['monto_total'] <= 0, 'monto_total'] = pd.NA

In [25]:
# comprobar si monto_total sigue la fórmula monto_total= precio_unitario*cantidad - monto_descuento
orders_work['monto_total_calculado']=(orders_work['precio_unitario']*orders_work['cantidad']-orders_work['monto_descuento'])

#verificamos las primeras filas
orders_work[['precio_unitario','cantidad','monto_descuento','monto_total_calculado','monto_total']].head()

,precio_unitario,cantidad,monto_descuento,monto_total_calculado,monto_total
0,332.69,2.0,0.0,665.38,665.37
1,176.86,1.0,5.0,171.86,171.86
2,102.99,2.0,10.0,195.98,195.99
3,257.87,1.0,15.0,242.87,242.87
4,336.28,1.0,0.0,336.28,336.28


In [26]:
# imputamos cantidad siguiendo el cálculo
orders_work['monto_total']=orders_work['monto_total'].fillna(orders_work['precio_unitario']*orders_work['cantidad']-orders_work['monto_descuento'])

In [27]:
# revisamos la imputación
cantidad_menores_igual_cero_monto_total = len(orders_work[orders_work['monto_total'] <= 0])
print(f"Cantidad de valores <= 0: {cantidad_menores_igual_cero_monto_total}")

Cantidad de valores <= 0: 0


In [28]:
#verificamos si existen duplicados en dataset orders (True/Flase por fila)
duplicados = orders_work.duplicated()
print(f"Cantidad de duplicados: {duplicados.sum()}")


Cantidad de duplicados: 100


In [29]:
#mostrar filas duplicadas
filas_duplicadas = orders_work[orders_work.duplicated(keep=False)]
print(filas_duplicadas)

         id_pedido id_usuario fecha_hora_pedido       pais dispositivo  \
734      order_734  user_6347        2025-05-02   Colombia     desktop   
812      order_812  user_1530        2025-03-31  Argentina     desktop   
974      order_974  user_3262        2025-05-04  Argentina     desktop   
1361    order_1361   user_507        2025-06-27     Mexico     desktop   
1735    order_1735   user_728        2025-05-13     Mexico      mobile   
...            ...        ...               ...        ...         ...   
25095   order_3913   user_380        2025-02-18  Argentina     desktop   
25096  order_23405  user_7833        2025-04-04   Colombia      mobile   
25097   order_5615  user_5417        2025-05-13   Colombia     desktop   
25098    order_812  user_1530        2025-03-31  Argentina     desktop   
25099   order_1361   user_507        2025-06-27     Mexico     desktop   

      fuente_referencia       nombre_producto categoria_producto cantidad  \
734             organic        Ble

In [30]:
#eliminar duplicados manteniendo el primero (por defecto)
orders_work = orders_work.drop_duplicates(keep='first').copy()

In [31]:
#verificar duplicados antes y después
print(f"Filas antes: {len(orders)}")
print(f"Filas después:{len(orders_work)}")

Filas antes: 25100
Filas después:25000


In [66]:
#verificamos si existen filas con el mismo id pedido
duplicados_id_pedido = orders_work[orders_work['id_pedido'].duplicated(keep=False)]

# Mostrar las filas duplicadas ordenadas por id_pedido
print(f"Cantidad de filas con id_pedido duplicado: {len(duplicados_id_pedido)}")
print("\nFilas con id_pedido duplicado:")
print(duplicados_id_pedido[['id_pedido', 'id_usuario', 'fecha_hora_pedido', 'pais', 'monto_total']])

Cantidad de filas con id_pedido duplicado: 0

Filas con id_pedido duplicado:
Empty DataFrame
Columns: [id_pedido, id_usuario, fecha_hora_pedido, pais, monto_total]
Index: []


In [32]:
# Revisar variables categóricas dataset orders de acuerdo a análisis exploratorio
# Cambiamos argentina por Argentina, mexico por Mexico y colombia por Colombia en columna pais dataset orders
orders_work['pais']=orders_work['pais'].replace({'argentina':'Argentina','mexico':'Mexico','colombia':'Colombia'})

In [33]:
# verificamos si los nombres se reemplazaron correctamente
orders_work['pais'].value_counts()

Mexico       8341
Colombia     8304
Argentina    8055
Name: pais, dtype: int64

In [34]:
# Convertir a número variables numéricas

def convertir_columnas_numericas(df,columnas): #creamos la función
    for col in columnas:
        df[col]=pd.to_numeric(df[col],errors='coerce')
    return df

#creamos la lista de columnas numéricas
col_numericas_orders_work=['cantidad','monto_total','valor_sin_dcto','cantidad_calculada','monto_total_calculado']

#aplicar la función convertir_columnas_numericas
orders_work= convertir_columnas_numericas(orders_work,col_numericas_orders_work)

#revisar resultado
orders_work.info()


<class 'pandas.core.frame.DataFrame'>
Int64Index: 25000 entries, 0 to 24999
Data columns (total 15 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   id_pedido              25000 non-null  object        
 1   id_usuario             25000 non-null  object        
 2   fecha_hora_pedido      25000 non-null  datetime64[ns]
 3   pais                   24700 non-null  object        
 4   dispositivo            24980 non-null  object        
 5   fuente_referencia      24970 non-null  object        
 6   nombre_producto        24970 non-null  object        
 7   categoria_producto     24920 non-null  object        
 8   cantidad               24946 non-null  float64       
 9   precio_unitario        24950 non-null  float64       
 10  monto_descuento        24950 non-null  float64       
 11  monto_total            24996 non-null  float64       
 12  valor_sin_dcto         24946 non-null  float64       
 13  c

**Resumen de limpieza de datasets:**
- Creamos copia de los datasets
- Validamos y convertimos fechas al formato correcto  
- Revisamos variables numéricas (sin negativos o ceros inválidos)  
- Verificamos consistencia de montos  
- Eliminamos duplicados  
- Revisamos variables categóricas
- Convertimos a número variables numéricas

---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [35]:
# exportar datasets
orders_work.to_csv('orders_clean.csv', index=False)
catalog_work.to_csv('catalog_clean.csv', index=False)
marketing_work.to_csv('marketing_clean.csv', index=False)

---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)? 
- ¿Cuál es el costo total? 
- ¿Cuánto se ha invertido en marketing? 
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden? 
- ¿Cuál es la cantidad promedio de productos por orden? 
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal? 

In [48]:
# Cargamos los archivos CSV 
orders_clean = pd.read_csv('orders_clean.csv')
catalog_clean = pd.read_csv('catalog_clean.csv')
marketing_clean = pd.read_csv('marketing_clean.csv')

In [58]:
#unimos las tablas orders con catalog con left join
merged_left = pd.merge(orders_clean,catalog_clean, on=['nombre_producto'],how='left')
merged_left

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto_x,cantidad,precio_unitario,monto_descuento,monto_total,valor_sin_dcto,cantidad_calculada,monto_total_calculado,categoria_producto_y,costo_unitario,proveedor
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37,665.38,2.0,665.38,Moda,189.31,Mcmillan-Rhodes
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86,176.86,1.0,171.86,Electrónica,25.21,Bowers LLC
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99,205.98,2.0,195.98,Hogar,176.64,Long-Reid
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87,257.87,1.0,242.87,Electrónica,25.21,Bowers LLC
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28,336.28,1.0,336.28,Hogar,176.64,Long-Reid
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24995,order_24995,user_4217,2025-04-16,Argentina,mobile,social,Tablet-Standard-64GB,Electronica,1.0,64.57,0.0,64.57,64.57,1.0,64.57,Electrónica,25.21,Bowers LLC
24996,order_24996,user_3945,2025-02-14,Argentina,mobile,social,Vacuum-Pro-Black,Hogar,2.0,292.90,15.0,570.81,585.80,2.0,570.80,Hogar,16.60,"Rivera, Carr and Finley"
24997,order_24997,user_3119,2025-03-27,Colombia,mobile,social,Jacket-Winter-M,Moda,1.0,226.85,0.0,226.85,226.85,1.0,226.85,Moda,189.31,Mcmillan-Rhodes
24998,order_24998,user_7218,2025-02-01,Mexico,mobile,social,Sneakers-Urban-42,Moda,2.0,418.06,0.0,836.12,836.12,2.0,836.12,Moda,17.21,Greene-Smith


In [61]:
marketing_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1620 entries, 0 to 1619
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   fecha       1620 non-null   object 
 1   pais        1620 non-null   object 
 2   id_campaña  1620 non-null   object 
 3   canal       1519 non-null   object 
 4   gasto       1620 non-null   float64
dtypes: float64(1), object(4)
memory usage: 63.4+ KB


In [62]:
#unimos la tabla merged_left con marketing con left join
merged_left_complete = pd.merge(merged_left, marketing_clean, 
                               left_on=['fecha_hora_pedido','pais','fuente_referencia'], 
                               right_on=['fecha','pais','canal'], 
                               how='left')
merged_left_complete

,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto_x,cantidad,precio_unitario,...,valor_sin_dcto,cantidad_calculada,monto_total_calculado,categoria_producto_y,costo_unitario,proveedor,fecha,id_campaña,canal,gasto
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,...,665.38,2.0,665.38,Moda,189.31,Mcmillan-Rhodes,2025-05-22,organic_Argentina,organic,2557.54
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,...,176.86,1.0,171.86,Electrónica,25.21,Bowers LLC,2025-06-15,paid_search_Mexico,paid_search,1021.85
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,...,205.98,2.0,195.98,Hogar,176.64,Long-Reid,2025-05-02,social_Argentina,social,2324.18
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,...,257.87,1.0,242.87,Electrónica,25.21,Bowers LLC,2025-06-09,social_Colombia,social,2854.66
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,...,336.28,1.0,336.28,Hogar,176.64,Long-Reid,2025-03-30,paid_search_Argentina,paid_search,1744.80
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24997,order_24995,user_4217,2025-04-16,Argentina,mobile,social,Tablet-Standard-64GB,Electronica,1.0,64.57,...,64.57,1.0,64.57,Electrónica,25.21,Bowers LLC,2025-04-16,social_Argentina,social,645.59
24998,order_24996,user_3945,2025-02-14,Argentina,mobile,social,Vacuum-Pro-Black,Hogar,2.0,292.90,...,585.80,2.0,570.80,Hogar,16.60,"Rivera, Carr and Finley",2025-02-14,social_Argentina,social,1557.65
24999,order_24997,user_3119,2025-03-27,Colombia,mobile,social,Jacket-Winter-M,Moda,1.0,226.85,...,226.85,1.0,226.85,Moda,189.31,Mcmillan-Rhodes,2025-03-27,social_Colombia,social,2345.12
25000,order_24998,user_7218,2025-02-01,Mexico,mobile,social,Sneakers-Urban-42,Moda,2.0,418.06,...,836.12,2.0,836.12,Moda,17.21,Greene-Smith,2025-02-01,social_Mexico,social,1511.83


Duplicados en marketing_clean:
Series([], dtype: int64)


In [65]:
# La nueva tabla contiene 25002 filas: dos filas más que la tabla orders_clean
# Agregamos un identificador único antes del merge para rastrear las dos filas demás.
merged_left_con_id = merged_left.copy()
merged_left_con_id['fila_original'] = range(len(merged_left_con_id))

# Hacer el merge con el identificador
merged_con_id = pd.merge(merged_left_con_id, marketing_clean,
                        left_on=['fecha_hora_pedido','pais','fuente_referencia'],
                        right_on=['fecha','pais','canal'],
                        how='left')

# Encontrar las filas duplicadas
filas_duplicadas = merged_con_id[merged_con_id['fila_original'].duplicated(keep=False)]
print("Filas que se duplicaron:")
print(filas_duplicadas[['id_pedido', 'fecha_hora_pedido', 'pais', 'fuente_referencia', 'fila_original']])

Filas que se duplicaron:
   id_pedido fecha_hora_pedido    pais fuente_referencia  fila_original
68  order_68        2025-01-22  Mexico               NaN             68
69  order_68        2025-01-22  Mexico               NaN             68
70  order_68        2025-01-22  Mexico               NaN             68


In [68]:
# Las filas duplicadas se producen porque el id_pedido order_68 de la tbla orders_clean, no contien un canal (NaN), pandas crea un registro por cada canal disponible.
# Eliminamos los duplicados y mantenemos solo uno
merged_left_complete = merged_left_complete.drop_duplicates(
    subset=['id_pedido', 'fecha_hora_pedido', 'pais'], 
    keep='first'
)

# Verificar el resultado
print(f"Filas antes: {len(merged_left_complete)}")

Filas antes: 25000


In [47]:
# Cálculo de KPI
ingreso_total=orders_clean['monto_total'].sum() # Ingreso Total
print(f"Ingreso_total: {ingreso_total:,.0f}".replace(',','.'))

costo_total=orders_clean['']

Ingreso_total: 51.988.610


In [59]:
marketing_clean.columns

Index(['fecha', 'pais', 'id_campaña', 'canal', 'gasto'], dtype='object')

In [44]:
orders_work.columns

Index(['id_pedido', 'id_usuario', 'fecha_hora_pedido', 'pais', 'dispositivo',
       'fuente_referencia', 'nombre_producto', 'categoria_producto',
       'cantidad', 'precio_unitario', 'monto_descuento', 'monto_total',
       'valor_sin_dcto', 'cantidad_calculada', 'monto_total_calculado'],
      dtype='object')

In [56]:
catalog_clean.columns

Index(['nombre_producto', 'categoria_producto', 'costo_unitario', 'proveedor'], dtype='object')

---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [ ]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [ ]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

In [ ]:
# PARTE 1: Totales del funnel
# ======================

query_totals = '''
tu código SQL aquí
'''

totals = pd.read_sql(query_totals, con=engine)
totals

In [ ]:
# PARTE 2: Conversiones
# ======================

query_conversion = '''
tu código SQL aquí
'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion

---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users` 
- `user_activity` 

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [ ]:
# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''
users = pd.read_sql(query_users, con=engine)
users.head(3)

In [ ]:
# Explorar tabla user_activity
# =========================
query_user_activity = '''
tu código SQL aquí
'''
user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)

In [ ]:
# Retención por cohortes
# ======================

query_cohort_retention_final = '''
tu código SQL aquí
'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final

---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado** 
4. **Interpretar el resultado**  

---
Hipótesis estadística
   - **H₀ (Hipótesis nula):** ...
   - **H₁ (Hipótesis alternativa):** ...
   
**Test estadístico:** ...  
**Nivel de significancia alpha:** ...

In [ ]:
# tu código aquí



---

## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión. 

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc

---

## 🚀 Entrega Final

Comparte el acceso a tu Dashboard para revisión.   
Puedes entregar el Dashboard utilizando **Power BI o Tableau**.

Incluye **uno de los siguientes**:

- 🔗 Link público del dashboard publicado en **Power BI Service o Tableau Public / Tableau Cloud**
- 🔗 Link de **Google Drive o OneDrive** con el archivo del proyecto (`.pbix`) y los 3 csvs limpios.


### 📎 Enlace del Dashboard

In [ ]:
# (Pega aquí tu link)
# link de power bi o tableau
# link de one drive / google drive